In [1]:
import pandas as pd
import os

df_raw = pd.read_csv('dataset.zip')
df = df_raw.copy()
print("Raw dataset loaded:", df_raw.shape)

Raw dataset loaded: (16598, 11)


## Phase 1 - Data Profiling
**Dataset:** Video Game Sales (Kaggle)
**Shape:** 16,598 rows x 11 column

### Problems Found
| Column | Issue | Count |
|---|---|---|
| Year | Missing values | 271 |
| Year | Wrong dtype(float64) | all rows |
| Publisher | Missing values | 58 |
No duplicate rows found.

In [2]:
# profiling the dataset ( phase 1 )
print( "Shape : " , df_raw.shape)
print("\n--- Data Types ---")
print(df_raw.dtypes)
print("\n--- Missing Values ---")
print(df_raw.isnull().sum())

print("\n--- Duplicate Rows ---")
print("Duplicates : " , df_raw.duplicated().sum())
print("\n--- Quick Stats ---")
df_raw.describe()

Shape :  (16598, 11)

--- Data Types ---
Rank              int64
Name             object
Platform         object
Year            float64
Genre            object
Publisher        object
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object

--- Missing Values ---
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

--- Duplicate Rows ---
Duplicates :  0

--- Quick Stats ---


,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


In [3]:
# phase 2a : Figuring out problems in column Year
print("Rows with Missing Years : ")
print(df[df['Year'].isnull()].shape[0])
print("\nYear range:")
print(df['Year'].min() , "to" , df["Year"].max())
print("\nSuspicious Years: ")
print(df[df['Year'] > 2016]['Year'].value_counts().sort_index())
print("\nSample of rows WHERE Year is missing: " )
df[df['Year'].isnull()][['Name', 'Platform', 'Genre', 'Publisher', 'Global_Sales']].head(10)

Rows with Missing Years : 
271

Year range:
1980.0 to 2020.0

Suspicious Years: 
Year
2017.0    3
2020.0    1
Name: count, dtype: int64

Sample of rows WHERE Year is missing: 


,Name,Platform,Genre,Publisher,Global_Sales
179,Madden NFL 2004,PS2,Sports,Electronic Arts,5.23
377,FIFA Soccer 2004,PS2,Sports,Electronic Arts,3.49
431,LEGO Batman: The Videogame,Wii,Action,Warner Bros. Interactive Entertainment,3.17
470,wwe Smackdown vs. Raw 2006,PS2,Fighting,NaN,3.00
607,Space Invaders,2600,Shooter,Atari,2.53
624,Rock Band,X360,Misc,Electronic Arts,2.48
649,Frogger's Adventures: Temple of the Frog,GBA,Adventure,Konami Digital Entertainment,2.39
652,LEGO Indiana Jones: The Original Adventures,Wii,Action,LucasArts,2.39
711,Call of Duty 3,Wii,Shooter,Activision,2.26
782,Rock Band,Wii,Misc,MTV Games,2.11


## Phase 2 - Cleaning the year Column

### Problems
- 271 missing values - dropped because we cannot reliably determinerelease year
- 4 rows with year > 2016 - dropped because the dataset was scrapped in 2016, sales data unreliably
- dtype was float64 - converted to int64 after nulls removed

### Decision 
Dropped 275 rows total(1.66% of data). Acceptable loss.

In [4]:
# phase 2b : Cleaining the column Year. Problems : missing years, suspicious post-2016 rows, incorrect data type in some values of Year.

df = df[df['Year'].notnull()] # dropping missing years rows
df = df[df['Year'] <= 2016]   # dropping post-2016 rows
df['Year'] = df['Year'].astype(int) # converting data type from float to int

# validating our clean data 
print("Missing Years now: " , df['Year'].isnull(). sum())
print("Year dtype now: " ,  df['Year'].dtype)
print("Rows remaining:",  df.shape[0])

Missing Years now:  0
Year dtype now:  int64
Rows remaining: 16323


## Phase 3 - Cleaning the Publisher Column

### Problems
- 36 missing values ( approximately 0.22% of data)

### Decision 
Droppped all 36 rows : too small to impute, publisher identity could not be reliable determined from game titles alone.


In [5]:
# phase 2c : Figuring out empty values in column Publisher.
print("Rows with missing Publisher: " )
print(df[df['Publisher'].isnull()].shape[0])
print("\nNames of Missing Publishers:")
df[df['Publisher'].isnull()]['Name']
# selecting multiple terms at the same time
(df[df['Publisher'].isnull()][['Name','Platform','Year','Genre','Global_Sales']].head(10))

Rows with missing Publisher: 
36

Names of Missing Publishers:


,Name,Platform,Year,Genre,Global_Sales
1662,Shrek / Shrek 2 2-in-1 Gameboy Advance Video,GBA,2007,Misc,1.21
2222,Bentley's Hackpack,GBA,2005,Misc,0.93
3159,Nicktoons Collection: Game Boy Advance Video V...,GBA,2004,Misc,0.64
3166,SpongeBob SquarePants: Game Boy Advance Video ...,GBA,2004,Misc,0.64
3766,SpongeBob SquarePants: Game Boy Advance Video ...,GBA,2004,Misc,0.53
4526,The Fairly Odd Parents: Game Boy Advance Video...,GBA,2004,Misc,0.43
4635,The Fairly Odd Parents: Game Boy Advance Video...,GBA,2004,Misc,0.42
5647,Cartoon Network Collection: Game Boy Advance V...,GBA,2005,Misc,0.32
6437,Sonic X: Game Boy Advance Video Volume 1,GBA,2004,Misc,0.27
6562,Dora the Explorer: Game Boy Advance Video Volu...,GBA,2004,Misc,0.26


In [6]:
#phase 2d
missing = df['Publisher'].isnull().sum() # the output will be 36
total = df.shape[0]
percentage = (missing/total) * 100
print(f"Missing Publisher : {missing} rows out of {total} = {percentage : .2f}%")

Missing Publisher : 36 rows out of 16323 =  0.22%


In [7]:
# dropping Missing Publisher rows ie keeping rows where publisher is not empty
df = df[df['Publisher'].notnull()]
print("Missing Publishers now: ", df['Publisher'].isnull().sum())
print("Rows remaining:", df.shape[0])

Missing Publishers now:  0
Rows remaining: 16287


## Phase 4 - Global Sales Consistency Check

### Finding
40.81% of rows showed Global_Sales \= (not equals) sum of regional sales.

### Decision
No action taken. Max difference was 0.02 million - confirmed floating point rounding artifact, not data corruption

In [8]:
# return false
total = df.shape[0]
incorrect_global_Sales = (df['Global_Sales'] != df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']).sum()
percentage = (incorrect_global_Sales / total) * 100
print(f"Incorrect global sales =  {incorrect_global_Sales}, Total Sales = {total} percentage = {percentage : .2f}%")

Incorrect global sales =  6646, Total Sales = 16287 percentage =  40.81%


In [9]:
df['Calculated_Sales'] = df['NA_Sales'] + df['EU_Sales'] + df['Other_Sales'] + df['JP_Sales']
# to check the difference
df['difference'] = abs(df['Global_Sales'] - df['Calculated_Sales'])
print(df['difference'].describe())


count    16287.000000
mean         0.002725
std          0.004467
min          0.000000
25%          0.000000
50%          0.000000
75%          0.010000
max          0.020000
Name: difference, dtype: float64


In [10]:
# dropping the temporary colums
df = df.drop(columns = ['Calculated_Sales', 'difference'])
print("Columns now: ", df.columns.tolist())

Columns now:  ['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']


In [11]:
# validating what we did
print("Total rows remaining :" , df.shape[0]) 
print("\nMissing rows in every column:" , df.isnull().sum())
print("\nData Type of year:")
print("\n Year Range : " , df['Year'].dtype)
print(df['Year'].min() , "TO", df['Year'].max())
print("\nUnique publishers : " , df['Publisher'].nunique())
print("\nUnique genres in coumn year and Global sales: " ,df[['Year','Global_Sales']].nunique())

Total rows remaining : 16287

Missing rows in every column: Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64

Data Type of year:

 Year Range :  int64
1980 TO 2016

Unique publishers :  576

Unique genres in coumn year and Global sales:  Year             37
Global_Sales    621
dtype: int64


In [12]:
import os
os.makedirs('data/cleaned',exist_ok=True)
os.makedirs('data/raw',exist_ok=True)

In [13]:
# saving the clean dataframe to a csv file
df.to_csv('data/cleaned/vgsales_clean.csv', index=False)
print("Exported!" , df.shape[0], "rows")

Exported! 16287 rows


In [ ]:
1